In [ ]:
# --- Librerias para manejo de datos ---
import pandas as pd          # Tablas de datos (DataFrames)
import numpy as np            # Operaciones matematicas

# --- Librerias de Machine Learning (scikit-learn) ---
from sklearn.model_selection import train_test_split   # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer          # Aplicar transformaciones por tipo de columna
from sklearn.preprocessing import OneHotEncoder        # Convertir categorias a numeros (0/1)
from sklearn.pipeline import Pipeline                  # Encadenar preprocesamiento + modelo
from sklearn.linear_model import LogisticRegression    # Modelo de clasificacion

# --- Metricas de evaluacion para clasificacion ---
from sklearn.metrics import (
    confusion_matrix,          # Matriz de confusion (tabla de aciertos y errores)
    classification_report,     # Reporte con Precision, Recall, F1-Score
    accuracy_score,            # Porcentaje de predicciones correctas
)

# --- Visualizacion ---
import matplotlib.pyplot as plt  # Graficos

print("Librerias cargadas correctamente.")

In [ ]:
# Cargar el archivo CSV en un DataFrame (tabla de datos)
df = pd.read_csv("abandono_clientes.csv")

# Mostrar las primeras 5 filas para verificar que se cargo correctamente
print("Primeras 5 filas del dataset:")
df.head()

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
df.describe().round(2)  # Estadisticas descriptivas para columnas numericas

In [ ]:
# DISTRIBUCION DE LA VARIABLE OBJETIVO
#
# ¿Por qué es importante revisar esto?
# En clasificación, las "clases" son los valores posibles de la variable objetivo.
# En este caso hay 2 clases: 0 (permanece) y 1 (abandona).
#
# "Balanceadas" significa que ambas clases tienen una cantidad similar de datos.
# Ejemplo balanceado:   50% permanece, 50% abandona → el modelo aprende bien ambas.
# Ejemplo desbalanceado: 99% permanece, 1% abandona → el modelo puede simplemente
# predecir siempre "permanece", obtener 99% de accuracy sin haber aprendido nada,
# y nunca detectar un cliente en riesgo → modelo inútil.
#
# Por eso revisamos la distribución antes de entrenar.
#

conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribución de la variable objetivo (abandono): ")
print(f" Permanece (0):  {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f" Abandona (1):  {conteo[1]} clientes ({porcentaje[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas")
print("Hay más clientes que permanecen que clientes que abandonan")
print("Esto es lo típico en la realidad: la mayoría de clientes NO se van.")

# Análisis exploratorio: como se comportan las variables según abandono?

Antes de entrenar el modelo, veamos si las variables tienen relación con el abandono. Esto nos ayuda a entender que factores podrían influir en la decisión del cliente.

In [ ]:
# Comparar el promedio de cada variable numérica según si el cliente abandonó o no

comparacion = df.groupby("abandono")[["antiguedad_meses", "factura_mensual", 
                                      "llamadas_soporte", "satisfaccion"]].mean().round(2)
comparacion.index = ["Permanece (0)", "Abandona (1)"]

print("Promedio de variables numéricas según abandono:")
comparacion

In [ ]:
# Análisis del tipo de contrato vs abandono

tabla_contrato = pd.crosstab(
    df["contrato"],
    df["abandono"],
    normalize="index"
).round(3) * 100

tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono por tipo de contrato:")
tabla_contrato

# Preparar los datos para el modelo

1. Separar variables predictoras (X) de la variable objetivo.
2. Entrenar los datos con (80%) del dataset y probar con el (20%) del dataset.

In [ ]:
# Paso 1: Separar X (variables predictoras) e y (variable objetivo)
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte"
        , "satisfaccion", "contrato"]]
y = df["abandono"]

# Paso 2: Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para prueba
    random_state=42,     # Semilla para reproducibilidad
    stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba: {X_test.shape[0]} clientes")
print("")
print(f"Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporcion de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("Las proporciones son similares gracias al parámetro 'stratify=y' en train_test_split")

# Construir el pipeline y entrenar el modelo

1. Calcula una puntuación lineal 
2. Pasa por esa puntuación que la convierte en una probabilidad entre 0 y 1.
3. Si la probabilidad es mayor a 0.5, predice la clase 1 (abandona): si no, predice 0 (permanece)

In [ ]:
# Definir columnas por tipo

numeric_features = ["antiguedad_meses", "factura_mensual", 
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]

# Preprocesamiento
preprocess = ColumnTransformer(
    transformers=[
    ("num", "passthrough", numeric_features),  
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]  
)

# Pipeline: preprocesamiento + modelo
# Garantizar que los datos siempre se preprocesen antes de llegar al modelo
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo
pipe.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

# Hacer predicciones

Usamos el modelo entrenado para predecir el abandono en los datos de prueba.

In [ ]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto": ["Si" if r == p else "No" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

# Evaluación del modelo: Exactitud/Presición (Acurracy)

Cuando hablamos del porcentaje de presición nos referimos al % de predicciones que fueron correctas

Accurracy = Predicciones Correctas/ Total de Predicciones

Ejemplo: si el 95% de clientes **NO** abandona, un modelo que siempre diga "NO abandona" tendría un 95% de presición, pero jamás detectaría a un cliente en riesgo. Sería inutil.

In [ ]:
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.1f}%%)")
print()
print(f"Esto significa que el modelo acertó {int(acc*len(y_test))} de {len(y_test)} predicciones.")

# Matriz de Confusión

La matriz de confusión nos muestra exactamente **dónde** acierta y **dónde** falla el modelo:

| | Predicho: Permanece | Predicho: Abandona |
|---|---|---|
| **Real: Permanece** | Verdadero Negativo (TN) | Falso Positivo (FP) |
| **Real: Abandona** | Falso Negativo (FN) | Verdadero Positivo (TP) |

- **Falso Negativo (FN)**: El modelo dijo "permanece" pero el cliente SÍ abandonó → el error más costoso.
- **Falso Positivo (FP)**: El modelo dijo "abandona" pero el cliente NO abandonó → menos grave.

In [ ]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Matriz de Confusión:")
print(f"  Verdaderos Negativos (TN): {tn} — permanece y el modelo dijo permanece")
print(f"  Falsos Positivos    (FP): {fp} — permanece pero el modelo dijo abandona")
print(f"  Falsos Negativos    (FN): {fn} — abandona pero el modelo dijo permanece")
print(f"  Verdaderos Positivos(TP): {tp} — abandona y el modelo dijo abandona")
print()
print(f"Errores totales: {fp + fn} de {len(y_test)} predicciones")

# Reporte de Clasificación

- **Precision**: De todos los que el modelo dijo "abandona", ¿cuántos realmente abandonaron?
- **Recall**: De todos los que realmente abandonaron, ¿cuántos detectó el modelo?
- **F1-Score**: Promedio armónico entre Precision y Recall (balance entre ambas).

In [ ]:
# Reporte de clasificación completo
print("Reporte de Clasificación:")
print()
print(classification_report(y_test, y_pred, 
                            target_names=["Permanece (0)", "Abandona (1)"]))

# Visualización de Resultados

Graficamos la matriz de confusión y la importancia de cada variable en la predicción del modelo.

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: Matriz de Confusión como heatmap ---
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Permanece", "Abandona"],
            yticklabels=["Permanece", "Abandona"],
            ax=axes[0], cbar=False,
            annot_kws={"size": 16})
axes[0].set_xlabel("Predicción del modelo", fontsize=12)
axes[0].set_ylabel("Valor real", fontsize=12)
axes[0].set_title("Matriz de Confusión", fontsize=14, fontweight="bold")

# --- Gráfica 2: Importancia de las variables (coeficientes del modelo) ---
# Obtener nombres de las variables después del preprocesamiento
cat_names = pipe.named_steps["preprocess"] \
    .transformers_[1][1].get_feature_names_out(["contrato"]).tolist()
feature_names = numeric_features + cat_names

coefs = pipe.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"Variable": feature_names, "Coeficiente": coefs})
coef_df = coef_df.sort_values("Coeficiente")

colors = ["#2ecc71" if c < 0 else "#e74c3c" for c in coef_df["Coeficiente"]]
axes[1].barh(coef_df["Variable"], coef_df["Coeficiente"], color=colors)
axes[1].set_xlabel("Coeficiente (peso en el modelo)", fontsize=12)
axes[1].set_title("Importancia de Variables", fontsize=14, fontweight="bold")
axes[1].axvline(x=0, color="gray", linestyle="--", linewidth=0.8)

# Leyenda
axes[1].text(0.98, 0.02, "Rojo = aumenta abandono\nVerde = reduce abandono",
             transform=axes[1].transAxes, fontsize=9,
             ha="right", va="bottom",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()

# Conclusiones

- El modelo de **Regresión Logística** alcanzó una exactitud del ~76%, lo cual es un buen punto de partida.
- Las variables que más influyen en el abandono son:
  - **Contrato mensual**: los clientes con contrato mensual tienen mayor probabilidad de abandonar.
  - **Antigüedad**: a mayor antigüedad, menor probabilidad de abandono.
  - **Satisfacción**: clientes con menor satisfacción tienden a abandonar más.
- La **matriz de confusión** permite ver si el modelo está fallando más en detectar abandonos (falsos negativos) o generando falsas alarmas (falsos positivos).